In [ ]:
import torch
import math
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import random
from torch import nn
from tqdm.auto import tqdm
import torchvision.transforms as T
import torch.nn.functional as F
import cv2
from torchvision.utils import make_grid
import os
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from PIL import Image
import random

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform = None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_folders = sorted(os.listdir(root_dir))
        self.length = len(os.listdir('Dataset/Images/Images/A'))

    def __len__(self):
        return self.length - 1

    def __getitem__(self, idx):
        example_images = []
        for letter_folder in self.image_folders:
            letter_folder_path = os.path.join(self.root_dir, letter_folder)
            image_files = sorted(os.listdir(letter_folder_path))
            image_path = os.path.join(letter_folder_path, image_files[idx])
            image = Image.open(image_path)
            if self.transform:
                image = self.transform(image)
            example_images.append(image)
        example = torch.stack(example_images, dim = 0)
        return example

stats = (0.5), (0.5)    
    
transform = T.Compose([
    T.Grayscale(num_output_channels = 1),
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize(*stats)
])


dataset = CustomDataset(root_dir = 'Dataset/Images/Images/', transform = transform)

In [ ]:
trained = False
epochs = 500
display_step = 50
img_channels = 26
crit_repeats = 5
gen_repeats = 1
learning_rate_D = 0.0002
learning_rate_G = 0.001
beta_1 = 0.0
beta_2 = 0.9
lambda_ = 10
z_dim = 100
device = 'cuda'
loss = 'W'
batch_size = 4

In [ ]:
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = False)

In [ ]:
for batch in dataloader:
    break

In [ ]:
batch.shape

In [ ]:
def show_tensor_images(image_tensor, num_images = 52, size = (1, 28, 28)):
    image_tensor = (image_tensor + 1) / 2
    image_unflat = image_tensor.detach().cpu()
    image_grid = make_grid(image_unflat[:num_images], nrow = 5)
    images = image_unflat[:num_images]
    plt.imshow(image_grid.permute(1, 2, 0).squeeze())
    plt.show()

In [ ]:
def generate_noise(n_examples, z_dim, device = 'cpu'):
    noise = torch.randn(n_examples, z_dim, device = device)
    return noise

In [ ]:
def weights_init(m):
    if isinstance(m, nn.Conv3d) or isinstance(m, nn.ConvTranspose3d):
        torch.nn.init.normal_(m.weight, 0.0, 0.02)
    if isinstance(m, nn.BatchNorm3d):
        torch.nn.init.normal_(m.weight, 0.0, 0.02)
        torch.nn.init.constant_(m.bias, 0)

In [ ]:
# class Generator(nn.Module):
#     def __init__(self, z_dim, n_channels):
#         super(Generator, self).__init__()
#         self.l1 = nn.Sequential(nn.Linear(z_dim, 128 * 1 * 8 * 8))

#         self.conv_blocks = nn.Sequential(
#             nn.BatchNorm3d(128),
#             nn.Upsample(scale_factor = (1, 2, 2)),
#             nn.Conv3d(128, 128, 3, stride = 1, padding = 1),
#             nn.BatchNorm3d(128, 0.8),
#             nn.ReLU(inplace = True),
#             nn.Upsample(scale_factor = (3, 2, 2)),
#             nn.Conv3d(128, 64, 3, stride = 1, padding = 1),
#             nn.BatchNorm3d(64, 0.8),
#             nn.ReLU(inplace = True),
#             nn.Conv3d(64, n_channels, 3, stride = 1, padding = 1),
#             nn.Tanh(),
#         )

#     def forward(self, z):
#         out = self.l1(z)
#         out = out.view(out.shape[0], 128, 1, 8, 8)
#         img = self.conv_blocks(out)
#         return img

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim = 10, img_channels = 1, hidden_dims = 16):
        super(Generator, self).__init__()
        self.z_dim = z_dim
        self.gen = nn.Sequential (
            self.make_gen_block(z_dim, img_channels, kernel_size = (18, 4, 4)),
            self.make_gen_block(img_channels, img_channels, kernel_size = (4, 4, 4), stride = (2, 2, 2)),
            self.make_gen_block(img_channels, img_channels, kernel_size = (4, 4, 4), stride = (2, 2, 2), padding = (1, 1, 1)),
            self.make_gen_block(img_channels, img_channels, kernel_size = (1, 4, 4), stride = (1, 2, 2), padding = (30, 1, 1)),
            self.make_gen_block(img_channels, img_channels, kernel_size = (2, 4, 4), stride = (1, 2, 2), padding = (2, 1, 1), final_layer = True)
        )
    def make_gen_block(self, input_channels, output_channels, kernel_size = (3, 4, 4), stride = (2, 2, 2), padding = 1, final_layer = False):
        if not final_layer:
            return nn.Sequential(
                nn.ConvTranspose3d(input_channels, output_channels, kernel_size, stride, padding),
                nn.BatchNorm3d(output_channels),
                nn.ReLU()
            )
        else:
            return nn.Sequential(
                nn.ConvTranspose3d(input_channels, output_channels, kernel_size, stride, padding),
                nn.Tanh()
            )
    def unsqueeze_noise(self, noise):
        return noise.view(len(noise), self.z_dim, 1, 1, 1)
    def forward(self, noise):
        noise_in = self.unsqueeze_noise(noise)
        return self.gen(noise_in)

In [ ]:
class Critic(torch.nn.Module):
    def __init__(self, in_channels = 1, out_conv_channels = 128):
        super(Critic, self).__init__()
        conv1_channels = int(out_conv_channels / 8)
        conv2_channels = int(out_conv_channels / 4)
        self.out_conv_channels = out_conv_channels

        self.conv1 = nn.Sequential(
            nn.Conv3d(
                in_channels = in_channels, out_channels = in_channels, kernel_size = (2, 4, 4),
                stride = (1, 2, 2), padding = (2, 1, 1), bias = False
            ),
            nn.BatchNorm3d(in_channels),
            nn.LeakyReLU(0.2, inplace = True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv3d(
                in_channels = in_channels, out_channels = in_channels, kernel_size = (1, 4, 4),
                stride = (1, 2, 2), padding = (2, 1, 1), bias = False
            ),
            nn.BatchNorm3d(in_channels),
            nn.LeakyReLU(0.2, inplace = True)
        )
        self.conv3 = nn.Sequential(
            nn.Conv3d(
                in_channels = in_channels, out_channels = 1, kernel_size = (4, 4, 4),
                stride = (2, 2, 2), padding = (0, 1, 1), bias = False
            ),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        return x.view(len(x), -1)

In [ ]:
def get_gradient(crit, real, fake, epsilon):
    interpolated_img = real * epsilon + fake * (1 - epsilon)
    pred = crit(interpolated_img)
    grad = torch.autograd.grad(
        inputs = interpolated_img,
        outputs = pred,
        grad_outputs = torch.ones_like(pred),
        create_graph = True,
        retain_graph = True
    )[0]
    return grad

In [ ]:
def gradient_penalty(gradient):
    gradient = gradient.view(len(gradient), -1)
    gradient_norm = gradient.norm(2, dim = 1)
    penalty = torch.mean((gradient_norm - 1) ** 2)
    return penalty

In [ ]:
def wasserstein_loss_gen(fake_pred):
    return -torch.mean(fake_pred)

In [ ]:
def wasserstein_loss_crit(fake_pred, real_pred, penalty, lambda_ = 0.1):
    crit_loss = torch.mean(fake_pred) - torch.mean(real_pred) + torch.mean(lambda_ * (penalty))
    return crit_loss

In [ ]:
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha = 0.25, gamma = 2.0, reduction = 'mean'):
        super(BinaryFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, input, target):
        bce_loss = F.binary_cross_entropy_with_logits(input, target, reduction = 'none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        if self.reduction == 'mean':
            return torch.mean(focal_loss)
        elif self.reduction == 'sum':
            return torch.sum(focal_loss)
        elif self.reduction == 'none':
            return focal_loss
        else:
            raise ValueError("Invalid reduction option. Use 'mean', 'sum', or 'none'.")

In [ ]:
def GAN(trained, epochs, display_step, img_channels, gen_repeats, crit_repeats, learning_rate_G, learning_rate_D, beta_1, beta_2, lambda_, z_dim, dataloader, loss, device):
#     gen = Generator(z_dim = z_dim, n_channels = img_channels).to(device)
    gen = Generator(z_dim = z_dim, img_channels = img_channels, hidden_dims = 16).to(device)
    crit = Critic(in_channels = img_channels).to(device)
    gen_optimizer = torch.optim.Adam(gen.parameters(), lr = learning_rate_G, betas = (beta_1, beta_2))
    crit_optimizer = torch.optim.Adam(crit.parameters(), lr = learning_rate_D, betas = (beta_1, beta_2))

    if trained == False:
        gen = gen.apply(weights_init)
        crit = crit.apply(weights_init)

    elif trained == True:
        gen.load_state_dict(torch.load('./Generator_' + str(loss) + '.pth'))
        crit.load_state_dict(torch.load('./Critic_' + str(loss) + '.pth'))
        gen_optimizer.load_state_dict(torch.load('./Generator_Optimizer_' + str(loss) + '.pth'))
        crit_optimizer.load_state_dict(torch.load('./Critic_Optimizer_' + str(loss) + '.pth'))
        gen.train()
        crit.train()

    if loss == 'BCE':
        criterion = nn.BCEWithLogitsLoss()
    elif loss == 'BFL':
        criterion = BinaryFocalLoss(alpha = 0.25, gamma = 2, reduction = 'mean')

    cur_step = 0
    generator_losses = []
    critic_losses = []
    for epoch in range(epochs):
        for real in tqdm(dataloader):
            cur_batch_size = len(real)
            real = real.to(device)
            mean_iteration_critic_loss = 0
            mean_iteration_generator_loss = 0
            if loss == 'W':
                for _ in range(crit_repeats):
                    crit_optimizer.zero_grad()
                    fake_noise = generate_noise(cur_batch_size, z_dim, device = device)
                    epsilon = torch.rand(len(real), 26, 1, 1, 1, device = device, requires_grad = True)
                    fake_imgs = gen(fake_noise)
                    fake_pred = crit(fake_imgs)
                    real_pred = crit(real)
                    grad = get_gradient(crit, real, fake_imgs, epsilon)
                    penalty = gradient_penalty(grad)
                    crit_loss = wasserstein_loss_crit(fake_pred, real_pred, penalty, lambda_)
                    mean_iteration_critic_loss += crit_loss.item() / crit_repeats
                    critic_losses += [mean_iteration_critic_loss]
                    crit_loss.backward(retain_graph = True)
                    torch.nn.utils.clip_grad_norm_(crit.parameters(), max_norm = 1.0)
                    crit_optimizer.step()
                cur_step += 1

                for _ in range(gen_repeats):
                    gen_optimizer.zero_grad()
                    fake_noise_1 = generate_noise(cur_batch_size, z_dim, device = device)
                    fake_imgs_1 = gen(fake_noise_1)
                    fake_pred_1 = crit(fake_imgs_1)
                    gen_loss = wasserstein_loss_gen(fake_pred_1)
                    mean_iteration_generator_loss += gen_loss.item() / gen_repeats
                    generator_losses += [mean_iteration_generator_loss]
                    gen_loss.backward(retain_graph = True)
                    gen_optimizer.step()

            elif loss == 'BCE' or loss == 'BFL':
                crit_optimizer.zero_grad()
                fake_noise = generate_noise(cur_batch_size, z_dim, device = device)
                fake_imgs = gen(fake_noise)
                fake_pred = crit(fake_imgs)
                real_pred = crit(real)
                crit_loss_fake = criterion(fake_pred, torch.zeros_like(fake_pred))
                crit_loss_real = criterion(real_pred, torch.ones_like(real_pred))
                crit_loss = (crit_loss_fake + crit_loss_real) / 2
                critic_losses += [crit_loss.item()]
                crit_loss.backward(retain_graph = True)
                crit_optimizer.step()
                cur_step += 1

                for _ in range(gen_repeats):
                    gen_optimizer.zero_grad()
                    fake_pred_1 = crit(fake_imgs)
                    gen_loss = criterion(fake_pred_1, torch.ones_like(fake_pred_1))
                    mean_iteration_generator_loss += gen_loss.item() / gen_repeats
                    generator_losses += [mean_iteration_generator_loss]
                    gen_loss.backward(retain_graph = True)
                    gen_optimizer.step()

            if cur_step % display_step == 0 and cur_step > 0:
                gen_mean = sum(generator_losses[-display_step:]) / display_step
                crit_mean = sum(critic_losses[-display_step:]) / display_step
                print(f"Epoch {epoch + 1}, step {cur_step}: Generator loss: {gen_mean}, critic loss: {crit_mean}")
                idxA = random.randint(0, len(real) - 1)
                idxB = random.randint(0, len(real) - 1)
                tAf = fake_imgs[idxA]
                tBf = fake_imgs[idxB]
                fake_imgs = torch.cat((tAf, tBf), axis = 0)
                tAr = real[idxA]
                tBr = real[idxB]
                real = torch.cat((tAr, tBr), axis = 0)
                show_tensor_images(fake_imgs, num_images = 52, size = (1, 32, 32))
                show_tensor_images(real, num_images = 52, size = (1, 32, 32))
                step_bins = 20
                if len(generator_losses) > len(critic_losses):
                    length = len(critic_losses)
                else:
                    length = len(generator_losses)
                num_examples = (length // step_bins) * step_bins
                plt.plot(
                    range(num_examples // step_bins),
                    torch.Tensor(generator_losses[:num_examples]).view(-1, step_bins).mean(1),
                    label="Generator Loss"
                )
                plt.plot(
                    range(num_examples // step_bins),
                    torch.Tensor(critic_losses[:num_examples]).view(-1, step_bins).mean(1),
                    label="Critic Loss"
                )
                plt.legend()
                plt.show()
        
        torch.save(gen.state_dict(), './Generator_' + str(loss) + '.pth')
        torch.save(crit.state_dict(), './Critic_' + str(loss) + '.pth')
        torch.save(gen_optimizer.state_dict(), './Generator_Optimizer_' + str(loss) + '.pth')
        torch.save(crit_optimizer.state_dict(), './Critic_Optimizer_' + str(loss) + '.pth')

    return gen, crit, gen_optimizer, crit_optimizer

In [ ]:
gen, crit, gen_opt, crit_opt = GAN(trained, epochs, display_step, img_channels, gen_repeats, crit_repeats, learning_rate_G, learning_rate_D, beta_1, beta_2, lambda_, z_dim, dataloader, loss, device)

In [ ]:
gen = Generator(z_dim = z_dim, img_channels = img_channels, hidden_dims = 16).to(device)
gen.load_state_dict(torch.load('Generator_W.pth'))

In [ ]:
noise = generate_noise(10, z_dim, device = device)
image_tensor = gen(noise)

In [ ]:
show_tensor_images(image_tensor[2], num_images = 26, size = (3, 32, 32))